# Buzy AI: Moteur de Raisonnement Métier avec Gemma 4

Buzy AI est un assistant d'intelligence artificielle conçu pour transformer des documents d'entreprise multimodaux en connaissances structurées et en recommandations décisionnelles explicables grâce à Gemma 4.

## Le problème

Les entreprises produisent quotidiennement des contrats, des factures, des comptes-rendus de réunion, des tableaux financiers et d'autres documents stratégiques.

Ces informations restent souvent dispersées dans différents formats, ce qui ralentit l'analyse et complique la prise de décision.

Répondre à des questions telles que :

- Quel fournisseur présente le plus grand risque ?
- Pourquoi le budget a-t-il été dépassé ?
- Faut-il renouveler un contrat ?

nécessite généralement plusieurs heures d'analyse manuelle.

## Solution

Buzy AI combines Gemma 4's multimodal understanding with lightweight knowledge extraction and reasoning to support business decision-making.

The system:

- extracts structured facts from heterogeneous documents,
- builds a lightweight business knowledge base,
- retrieves relevant evidence,
- produces transparent recommendations supported by explicit reasoning.

## Architecture du Workflow

```mermaid
graph TD
    A[Documents<br><i>Factures, Bons de commande, Contrats, Images</i>] --> B[Gemma 4 Vision<br><i>Traitement multimodal document & image</i>]
    B --> C[Fact Extraction<br><i>Extraction JSON & Champs structurés</i>]
    C --> D[Knowledge Base<br><i>Stockage vectoriel & Base relationnelle</i>]
    D --> E[Reasoning Engine<br><i>Raisonnement logique & Règles d'affaires</i>]
    E --> F[Business Decision<br><i>Approbation, Alerte de fraude, Paiement</i>]

## Vue d'ensemble du pipeline

Ce notebook présente un pipeline complet de raisonnement métier :

1. Chargement de Gemma 4
2. Analyse de documents multimodaux
3. Extraction de connaissances structurées
4. Construction d'une base de connaissances
5. Adaptation du modèle par LoRA
6. Recherche des informations pertinentes
7. Raisonnement métier explicable
8. Démonstration d'un agent IA

## Install Dependencies

Install the libraries required for Gemma 4, Unsloth, LoRA fine-tuning, and data processing.

In [1]:
# Install dependencies
try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
except: _numpy = "numpy"; _pil = "pillow"
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes \
    unsloth "unsloth_zoo>=2026.4.5" transformers==5.5.0 torchcodec timm pandas scikit-learn

## Load Gemma 4

Load the quantized Gemma 4 instruction model optimized for efficient inference on Kaggle GPUs.

In [2]:
import os, gc, torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Libère la mémoire GPU au cas où une exécution précédente aurait laissé des tenseurs en mémoire
gc.collect()
torch.cuda.empty_cache()

from unsloth import FastModel

max_seq_length = 1024  # réduit vs 2048 pour laisser plus de marge mémoire sur un T4

model, processor = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-E2B-it-unsloth-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
    full_finetuning=False,
    device_map={"": 0},
    use_gradient_checkpointing="unsloth",
)

tokenizer = processor

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.5: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4 won't work! Using float32.


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

In [3]:
from transformers import TextStreamer

def do_gemma_4_inference(messages, max_new_tokens=350):
    _ = model.generate(
        **tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to("cuda"),
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_p=0.9,
        top_k=64,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
        use_cache=True,
    )

import torch, json, re

def generate_text(messages, max_new_tokens=350):
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to("cuda")
    input_length = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    return tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()

## Documents d'entreprise

Le pipeline utilise un ensemble représentatif de documents métiers comprenant :

- des contrats,
- des factures,
- des comptes-rendus de réunion,
- des données budgétaires.

Ces documents servent de base à la construction des connaissances exploitées par le moteur de raisonnement.

In [4]:
business_documents = [
    {
        "id": "DOC-CONTRACT-ATLAS",
        "type": "contract",
        "format": "text",
        "content": (
            "CONTRAT DE FOURNITURE - Fournisseur: Atlas Components Ltd. "
            "Date de signature: 2026-01-14. Durée: 6 mois, renouvelable. "
            "Date d'expiration actuelle: 2026-08-21. "
            "Conditions de paiement: 30 jours net. "
            "Clause de pénalité de retard: 2% par semaine de retard de livraison au-delà de 10 jours."
        ),
    },
    {
        "id": "DOC-INVOICE-1",
        "type": "invoice",
        "format": "text",
        "content": (
            "Facture INV-4471 - Fournisseur: Atlas Components Ltd. "
            "Date d'émission: 2026-05-02. Date d'échéance: 2026-06-01. Date de paiement réel: 2026-06-19. "
            "Retard de paiement: 18 jours. Montant: 42 300 USD."
        ),
    },
    {
        "id": "DOC-INVOICE-2",
        "type": "invoice",
        "format": "text",
        "content": (
            "Facture INV-4502 - Fournisseur: Atlas Components Ltd. "
            "Date d'émission: 2026-06-10. Date d'échéance: 2026-07-10. Date de paiement réel: 2026-07-28. "
            "Retard de paiement: 18 jours. Montant: 38 750 USD."
        ),
    },
    {
        "id": "DOC-MEETING-1",
        "type": "meeting_notes",
        "format": "text",
        "content": (
            "Compte-rendu réunion Achats - 2026-05-20. Participants: équipe Procurement. "
            "Point discuté: instabilité de livraison du fournisseur Atlas mentionnée pour la 2e fois ce "
            "trimestre. Le Projet Phoenix dépend exclusivement de ce fournisseur pour le composant X."
        ),
    },
    {
        "id": "DOC-MEETING-2",
        "type": "meeting_notes",
        "format": "text",
        "content": (
            "Compte-rendu réunion Achats - 2026-06-25. Instabilité de livraison Atlas de nouveau évoquée. "
            "Le Projet Nova utilise également des composants Atlas, dépendance secondaire identifiée."
        ),
    },
    {
        "id": "DOC-BUDGET",
        "type": "spreadsheet",
        "format": "text",
        "content": (
            "Ligne budgétaire Achats Q2 2026 - Budget prévu: 180 000 USD. Dépense réelle: 212 400 USD. "
            "Dépassement: 18%. Cause principale mentionnée par l'équipe Finance: pénalités de retard "
            "fournisseur et achats de secours en urgence."
        ),
    },
]

print(f"{len(business_documents)} documents chargés.")

6 documents chargés.


## Compréhension multimodale

Grâce aux capacités Vision-Language de Gemma 4, le système peut analyser directement des documents numérisés et convertir leur contenu en données structurées.

In [5]:
invoice_image_url = "/kaggle/input/datasets/destinbir1/invoices/Screenshot from 2026-07-25 08-49-30.png"

vision_extraction_messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": invoice_image_url},
        {"type": "text", "text":
            "Extract the following fields from this invoice image as strict JSON only "
            "(no explanation): {\"supplier\": ..., \"invoice_id\": ..., \"due_date\": ..., "
            "\"amount\": ..., \"payment_date\": ...}. If a field is not visible, use null."}
    ]
}]

do_gemma_4_inference(vision_extraction_messages, max_new_tokens=200)

```json
{
  "supplier": "LOGISTIQUE & FOURNITURES SARL",
  "invoice_id": "INV-8942-FR",
  "due_date": "18 Mai 2026",
  "amount": "650,00 €",
  "payment_date": "20 Mai 2026"
}
```<turn|>


## Extraction des connaissances

Chaque document est analysé afin d'identifier les entités, événements, risques, informations contractuelles, indicateurs financiers et dépendances opérationnelles.

Les informations extraites sont converties dans un format structuré exploitable par le moteur de raisonnement.

## Construction de la base de connaissances

Les faits extraits sont regroupés dans une base de connaissances légère permettant de retrouver rapidement les informations pertinentes avant le raisonnement.

Cette approche facilite l'explicabilité des décisions générées par le modèle.

In [6]:
import pandas as pd

def extract_facts_with_gemma(document):
    prompt = f"""
Extract structured business facts from the following document as a JSON list.
Each fact must follow this schema:
{{"entity": "...", "fact_type": "...", "value": "...", "date": "YYYY-MM-DD or null", "source_doc": "{document['id']}"}}

fact_type must be one of: payment_delay, contract_expiry, project_dependency, delivery_risk, budget_variance, other

DOCUMENT ({document['type']}):
{document['content']}

Return ONLY the JSON list, no explanation.
""".strip()

    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    raw_output = generate_text(messages, max_new_tokens=400)

    json_match = re.search(r"\[.*\]", raw_output, flags=re.DOTALL)
    if not json_match:
        print(f"[!] Pas de JSON valide pour {document['id']}. Sortie brute : {raw_output[:200]}")
        return []
    try:
        return json.loads(json_match.group())
    except json.JSONDecodeError:
        print(f"[!] JSON invalide pour {document['id']}.")
        return []

all_facts = []
for doc in business_documents:
    facts = extract_facts_with_gemma(doc)
    all_facts.extend(facts)
    print(f"{doc['id']}: {len(facts)} faits extraits")

knowledge_table = pd.DataFrame(all_facts)
knowledge_table

DOC-CONTRACT-ATLAS: 5 faits extraits
DOC-INVOICE-1: 1 faits extraits
DOC-INVOICE-2: 1 faits extraits
DOC-MEETING-1: 2 faits extraits
DOC-MEETING-2: 3 faits extraits
DOC-BUDGET: 3 faits extraits


,entity,fact_type,value,date,source_doc
0,Atlas Components Ltd.,contract_signature,2026-01-14,2026-01-14,DOC-CONTRACT-ATLAS
1,Contract Duration,contract_expiry,"6 mois, renouvelable",None,DOC-CONTRACT-ATLAS
2,Contract Expiration Date,contract_expiry,2026-08-21,2026-08-21,DOC-CONTRACT-ATLAS
3,Payment Terms,other,30 jours net,None,DOC-CONTRACT-ATLAS
4,Delivery Penalty Clause,payment_delay,2% par semaine de retard de livraison au-delà ...,None,DOC-CONTRACT-ATLAS
5,Invoice INV-4471,payment_delay,18 days late payment,2026-06-19,DOC-INVOICE-1
6,Invoice INV-4502,payment_delay,18 days late payment,2026-07-28,DOC-INVOICE-2
7,Atlas supplier,delivery_risk,instability of delivery mentioned for the seco...,2026-05-20,DOC-MEETING-1
8,Project Phoenix,project_dependency,depends exclusively on the Atlas supplier for ...,None,DOC-MEETING-1
9,Atlas delivery,delivery_risk,Instability of delivery mentioned,2026-06-25,DOC-MEETING-2


## Jeu de ddonnées de raisonnement

L'objectif du fine-tuning n'est pas d'apprendre davantage de connaissances métier, mais d'apprendre une méthode de raisonnement cohérente.

Chaque réponse suit systématiquement la structure suivante :

- preuves utilisées ;
- raisonnement ;
- niveau de confiance ;
- impact métier ;
- recommandations.

In [7]:
def make_example(question, context_facts, evidence, reasoning, confidence, impact, actions):
    context_text = "\n".join(f"- {f}" for f in context_facts)
    user_text = f"""Business question: {question}

Relevant facts retrieved:
{context_text}

Answer using ONLY the JSON schema:
{{"evidence": [...], "reasoning": [...], "confidence": 0-1, "business_impact": "...", "recommended_actions": [...]}}"""

    answer = json.dumps({
        "evidence": evidence,
        "reasoning": reasoning,
        "confidence": confidence,
        "business_impact": impact,
        "recommended_actions": actions,
    }, ensure_ascii=False)

    return {"conversations": [
        {"role": "user", "content": user_text},
        {"role": "model", "content": answer},
    ]}

reasoning_dataset_raw = [
    make_example(
        question="Which supplier represents the highest operational risk next quarter?",
        context_facts=[
            "Invoices from Atlas Components paid 18 days late on average over the last 3 months",
            "Two projects (Phoenix, Nova) depend on Atlas Components",
            "Atlas contract expires in 27 days",
            "Delivery instability from Atlas mentioned in 2 procurement meetings",
        ],
        evidence=[
            "Average payment delay of 18 days across 2 recent invoices (INV-4471, INV-4502)",
            "Contract expiration in 27 days",
            "Delivery instability mentioned in meetings on 2026-05-20 and 2026-06-25",
            "Project Phoenix depends exclusively on this supplier",
        ],
        reasoning=[
            "Repeated payment delays combined with repeated delivery instability indicate a supplier "
            "relationship under strain, not a one-off incident",
            "The contract expiring in under a month creates urgency: any renegotiation or supplier switch "
            "needs to start now",
            "Because Project Phoenix has no alternative supplier, disruption would directly delay a live project",
        ],
        confidence=0.82,
        impact="Estimated 2-week delay risk to Project Phoenix if supply is disrupted; potential renewal "
               "at unfavorable terms if negotiations start late",
        actions=[
            "Begin renewal negotiations with Atlas immediately",
            "Qualify a secondary supplier for the component used in Project Phoenix",
            "Escalate payment approval process before the renewal deadline",
        ],
    ),
    make_example(
        question="Why did the Q2 procurement budget exceed forecast?",
        context_facts=[
            "Q2 procurement budget: planned 180,000 USD, actual 212,400 USD (18% over)",
            "Finance notes late-delivery penalties and emergency backup purchases as main drivers",
        ],
        evidence=[
            "18% budget overrun (32,400 USD) recorded in Q2 spreadsheet",
            "Finance team attributes the variance to supplier late-delivery penalties and emergency purchases",
        ],
        reasoning=[
            "The overrun is not due to increased demand or pricing, but to reactive spending caused by "
            "supplier reliability issues",
            "This links directly to the supplier risk pattern observed for Atlas Components in the same period",
        ],
        confidence=0.75,
        impact="Recurring supplier instability could cause similar or larger overruns in Q3 if unaddressed",
        actions=[
            "Track penalty and emergency-purchase costs as a separate budget line to monitor supplier risk cost",
            "Address root cause (supplier reliability) rather than only adjusting the forecast",
        ],
    ),
    make_example(
        question="Should we renew the Atlas Components contract?",
        context_facts=[
            "Atlas contract expires in 27 days",
            "Average payment delay 18 days on last 2 invoices",
            "Delivery instability mentioned twice in procurement meetings",
            "No qualified alternative supplier currently identified",
        ],
        evidence=[
            "Contract expiration in 27 days with no alternative supplier qualified yet",
            "Pattern of delivery instability and payment friction over the last quarter",
        ],
        reasoning=[
            "Switching suppliers abruptly without a qualified alternative would be riskier than a short "
            "renewal with revised terms",
            "The recurring issues justify renegotiating terms (penalties, SLAs) rather than a plain renewal",
        ],
        confidence=0.68,
        impact="A short-term renewal with stronger SLAs reduces risk while a secondary supplier is qualified, "
               "avoiding an abrupt supply gap for Project Phoenix",
        actions=[
            "Negotiate a short-term renewal (e.g. 3 months) with stricter delivery and payment SLAs",
            "Run supplier qualification for an alternative in parallel",
            "Revisit the decision once a second supplier is validated",
        ],
    ),
    make_example(
        question="What is driving the increase in customer complaints this month?",
        context_facts=[
            "Support tickets mention 'late delivery' in 60% of complaints this month, up from 22% last month",
            "Delivery delays correlate with the same period as Atlas Components' late shipments",
        ],
        evidence=[
            "Late-delivery complaints rose from 22% to 60% of tickets month-over-month",
            "Timing overlaps with Atlas Components delivery instability",
        ],
        reasoning=[
            "The sharp increase in a single complaint category, aligned in time with a known supplier issue, "
            "suggests a shared root cause rather than unrelated customer service problems",
        ],
        confidence=0.7,
        impact="Continued delivery delays risk customer churn if not resolved before next renewal cycle",
        actions=[
            "Proactively notify affected customers of delivery timelines",
            "Prioritize resolving the underlying supplier delay issue",
        ],
    ),
]

print(f"{len(reasoning_dataset_raw)} exemples de raisonnement structuré.")

4 exemples de raisonnement structuré.


In [8]:
from datasets import Dataset
from unsloth.chat_templates import get_chat_template, standardize_data_formats

tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

reasoning_dataset = Dataset.from_list(reasoning_dataset_raw)
reasoning_dataset = standardize_data_formats(reasoning_dataset, aliases_for_assistant=["model"])

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False).removeprefix("<bos>")
        for convo in convos
    ]
    return {"text": texts}

reasoning_dataset = reasoning_dataset.map(formatting_prompts_func, batched=True)
reasoning_dataset[0]["text"]

num_proc must be <= 4. Reducing num_proc to 4 for dataset of size 4.
[datasets.arrow_dataset|WARNING]num_proc must be <= 4. Reducing num_proc to 4 for dataset of size 4.


Unsloth: Standardizing formats (num_proc=4):   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

'<|turn>user\nBusiness question: Which supplier represents the highest operational risk next quarter?\n\nRelevant facts retrieved:\n- Invoices from Atlas Components paid 18 days late on average over the last 3 months\n- Two projects (Phoenix, Nova) depend on Atlas Components\n- Atlas contract expires in 27 days\n- Delivery instability from Atlas mentioned in 2 procurement meetings\n\nAnswer using ONLY the JSON schema:\n{"evidence": [...], "reasoning": [...], "confidence": 0-1, "business_impact": "...", "recommended_actions": [...]}<turn|>\n<|turn>model\n{"evidence": ["Average payment delay of 18 days across 2 recent invoices (INV-4471, INV-4502)", "Contract expiration in 27 days", "Delivery instability mentioned in meetings on 2026-05-20 and 2026-06-25", "Project Phoenix depends exclusively on this supplier"], "reasoning": ["Repeated payment delays combined with repeated delivery instability indicate a supplier relationship under strain, not a one-off incident", "The contract expiring 

## Fine-tuning avec LoRA

Gemma 4 est adapté grâce à LoRA (Low-Rank Adaptation), une méthode efficace permettant de spécialiser le modèle tout en limitant le nombre de paramètres entraînés.

In [9]:
from peft import PeftModelForCausalLM

if not isinstance(model, PeftModelForCausalLM):
    model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 8,
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    )

In [10]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = reasoning_dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 2,
        num_train_epochs = 20,  
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)

num_proc must be <= 4. Reducing num_proc to 4 for dataset of size 4.
[datasets.arrow_dataset|WARNING]num_proc must be <= 4. Reducing num_proc to 4 for dataset of size 4.


Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/4 [00:00<?, ? examples/s]

In [11]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [12]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4 | Num Epochs = 20 | Total steps = 20
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 12,668,928 of 5,135,846,944 (0.25% trained)


Step,Training Loss
1,0.916565
2,0.916565
3,0.888891
4,0.786313
5,0.697784
6,0.623999
7,0.559467
8,0.504366
9,0.459365
10,0.425356


## Recherche des informations pertinentes

Avant de répondre à une question métier, le système récupère automatiquement les faits les plus pertinents présents dans la base de connaissances.

Ces informations servent de contexte au raisonnement du modèle.

In [13]:
def retrieve_facts(keyword, table=knowledge_table, top_k=8):
    if table.empty:
        return []
    mask = table.apply(lambda row: keyword.lower() in str(row.values).lower(), axis=1)
    matched = table[mask]
    return matched.head(top_k).to_dict(orient="records")

retrieve_facts("Atlas")

[{'entity': 'Atlas Components Ltd.',
  'fact_type': 'contract_signature',
  'value': '2026-01-14',
  'date': '2026-01-14',
  'source_doc': 'DOC-CONTRACT-ATLAS'},
 {'entity': 'Contract Duration',
  'fact_type': 'contract_expiry',
  'value': '6 mois, renouvelable',
  'date': None,
  'source_doc': 'DOC-CONTRACT-ATLAS'},
 {'entity': 'Contract Expiration Date',
  'fact_type': 'contract_expiry',
  'value': '2026-08-21',
  'date': '2026-08-21',
  'source_doc': 'DOC-CONTRACT-ATLAS'},
 {'entity': 'Payment Terms',
  'fact_type': 'other',
  'value': '30 jours net',
  'date': None,
  'source_doc': 'DOC-CONTRACT-ATLAS'},
 {'entity': 'Delivery Penalty Clause',
  'fact_type': 'payment_delay',
  'value': '2% par semaine de retard de livraison au-delà de 10 jours',
  'date': None,
  'source_doc': 'DOC-CONTRACT-ATLAS'},
 {'entity': 'Atlas supplier',
  'fact_type': 'delivery_risk',
  'value': 'instability of delivery mentioned for the second time this quarter',
  'date': '2026-05-20',
  'source_doc': 'DO

## Raisonnement métier explicable

À partir des informations récupérées, Gemma 4 produit une décision argumentée comprenant :

- les preuves utilisées ;
- le raisonnement suivi ;
- un niveau de confiance ;
- l'impact métier estimé ;
- les actions recommandées.

Cette approche favorise des décisions transparentes et vérifiables.

In [14]:
def business_reasoning(question, keyword):
    facts = retrieve_facts(keyword)
    if not facts:
        print("Aucun fait pertinent trouvé dans la base de connaissances pour cette question.")
        return None

    facts_text = "\n".join(f"- {f}" for f in facts)

    prompt = f"""Business question: {question}

Relevant facts retrieved:
{facts_text}

Answer using ONLY the JSON schema:
{{"evidence": [...], "reasoning": [...], "confidence": 0-1, "business_impact": "...", "recommended_actions": [...]}}"""

    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    raw_output = generate_text(messages, max_new_tokens=400)

    json_match = re.search(r"\{.*\}", raw_output, flags=re.DOTALL)
    if not json_match:
        print("Sortie brute (non JSON) :", raw_output)
        return None

    result = json.loads(json_match.group())
    print(json.dumps(result, indent=2, ensure_ascii=False))
    return result


result = business_reasoning(
    question="Which supplier represents the highest operational risk next quarter?",
    keyword="Atlas",
)

{
  "evidence": [
    "Atlas supplier delivery risk noted twice this quarter",
    "Project Phoenix depends exclusively on Atlas supplier for component X"
  ],
  "reasoning": [
    "The supplier Atlas Components Ltd. has demonstrated repeated delivery instability, and they are a sole-source supplier for a critical component (Component X) needed for Project Phoenix. This combination of repeated risk and single-source dependency elevates their operational risk profile."
  ],
  "confidence": 0.8,
  "business_impact": "Disruption to Project Phoenix timeline and potential production delays if Atlas delivery is delayed.",
  "recommended_actions": [
    "Increase buffer stock for components supplied by Atlas",
    "Qualify a secondary supplier for Component X by Q3"
  ]
}


## Agent IA

Le système peut également interagir avec des fonctions métier afin de récupérer automatiquement des informations complémentaires avant de formuler sa réponse.

Cette approche rapproche le modèle d'un véritable assistant décisionnel connecté aux outils de l'entreprise.

In [15]:
def get_invoice_status(supplier):
    rows = knowledge_table[
        (knowledge_table.get("entity", "").astype(str).str.contains(supplier, case=False, na=False))
        & (knowledge_table.get("fact_type", "") == "payment_delay")
    ]
    if rows.empty:
        return f"Aucun retard de paiement enregistré pour {supplier}."
    return rows.to_dict(orient="records")

def get_contract_expiry(supplier):
    rows = knowledge_table[
        (knowledge_table.get("entity", "").astype(str).str.contains(supplier, case=False, na=False))
        & (knowledge_table.get("fact_type", "") == "contract_expiry")
    ]
    if rows.empty:
        return f"Aucune échéance de contrat trouvée pour {supplier}."
    return rows.to_dict(orient="records")

business_tools = {"get_invoice_status": get_invoice_status, "get_contract_expiry": get_contract_expiry}


def run_business_agent(user_question):
    agent_prompt = f"""You are an agent with access to these tools:

get_invoice_status(supplier)   -> payment delay history for a supplier
get_contract_expiry(supplier)  -> contract expiration facts for a supplier

USER QUESTION:
{user_question}

Do not answer directly. Return ONLY JSON in this format:
{{"function": "tool_name", "arguments": {{"supplier": "..."}}}}""".strip()

    agent_messages = [{"role": "user", "content": [{"type": "text", "text": agent_prompt}]}]
    raw_output = generate_text(agent_messages, max_new_tokens=100)

    json_match = re.search(r"\{.*\}", raw_output, flags=re.DOTALL)
    if not json_match:
        print("Pas de JSON valide. Sortie brute :", raw_output)
        return

    tool_call = json.loads(json_match.group())
    function_name = tool_call.get("function")
    supplier = tool_call.get("arguments", {}).get("supplier", "")

    if function_name not in business_tools:
        print("Fonction non autorisée :", function_name)
        return

    tool_result = business_tools[function_name](supplier)
    print(f"[Agent] {function_name}({supplier!r}) -> {tool_result}\n")

    final_prompt = f"""User question: {user_question}
Tool result: {tool_result}

Answer in 1-2 clear sentences."""
    final_messages = [{"role": "user", "content": [{"type": "text", "text": final_prompt}]}]
    do_gemma_4_inference(final_messages, max_new_tokens=120)


run_business_agent("When does the Atlas contract expire?")

[Agent] get_contract_expiry('Atlas') -> Aucune échéance de contrat trouvée pour Atlas.

The provided information does not contain an expiration date for the Atlas contract. Therefore, I cannot tell you when it expires.<turn|>


## Raisonnement multilingue

Le pipeline conserve le même niveau de qualité de raisonnement quelle que soit la langue de la question, facilitant son utilisation dans des environnements professionnels multilingues.

In [16]:
def business_reasoning_multilingual(question, keyword, answer_language="français"):
    facts = retrieve_facts(keyword)
    if not facts:
        print("Aucun fait pertinent trouvé.")
        return None

    facts_text = "\n".join(f"- {f}" for f in facts)
    prompt = f"""Business question: {question}

Relevant facts:
{facts_text}

Answer in {answer_language}. Use ONLY this JSON schema:
{{"evidence": [...], "reasoning": [...], "confidence": 0-1, "business_impact": "...", "recommended_actions": [...]}}"""

    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    do_gemma_4_inference(messages, max_new_tokens=400)

business_reasoning_multilingual(
    "Quel fournisseur représente le plus grand risque opérationnel le trimestre prochain ?",
    keyword="Atlas",
    answer_language="français",
)

```json
{
  "evidence": [
    {
      "fact_id": "Atlas supplier",
      "value": "instability of delivery mentioned for the second time this quarter",
      "date": "2026-05-20",
      "source_doc": "DOC-MEETING-1"
    },
    {
      "fact_id": "Atlas delivery",
      "value": "Instability of delivery mentioned",
      "date": "2026-06-25",
      "source_doc": "DOC-MEETING-2"
    },
    {
      "fact_id": "Project Phoenix",
      "value": "depends exclusively on the Atlas supplier for component X",
      "date": null,
      "source_doc": "DOC-MEETING-1"
    }
  ],
  "reasoning": [
    "Les faits indiquent que le fournisseur Atlas Components Ltd. a fait l'objet de deux mentions concernant l'instabilité de la livraison au cours du trimestre précédent (Mai et Juin 2026).",
    "De plus, le projet 'Project Phoenix' dépend exclusivement de ce fournisseur pour un composant critique (component X), ce qui rend toute perturbation de la livraison un risque direct pour ce projet.",
    "Bien qu'

## Sauvegarde du modèle

Les poids LoRA entraînés peuvent être sauvegardés afin de faciliter leur réutilisation, leur déploiement ou leur partage.

In [17]:
model.save_pretrained("buzy_ai_reasoning_lora")
tokenizer.save_pretrained("buzy_ai_reasoning_lora")
# model.push_to_hub("HF_ACCOUNT/buzy-ai-gemma4-lora", token="YOUR_HF_TOKEN")
# tokenizer.push_to_hub("HF_ACCOUNT/buzy-ai-gemma4-lora", token="YOUR_HF_TOKEN")

Unsloth: Restored added_tokens_decoder metadata in buzy_ai_reasoning_lora/tokenizer_config.json.


['buzy_ai_reasoning_lora/processor_config.json']

In [ ]:
# Merge LoRA weights into the base model for standalone deployment
import torch
from peft import PeftModel
from unsloth import FastModel

# Load the base model in full precision (not quantized) for proper merging
base_model, processor = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-E2B-it-unsloth-bnb-4bit",
    max_seq_length=1024,
    dtype=torch.bfloat16,
    load_in_4bit=False,
    device_map="auto",
)

# Load the fine-tuned LoRA adapter
model = PeftModel.from_pretrained(base_model, "buzy_ai_reasoning_lora")

# Merge LoRA weights into the base model
merged_model = model.merge_and_unload()

# Save the merged model locally
merged_model.save_pretrained("buzy_ai_gemma4_merged")
processor.save_pretrained("buzy_ai_gemma4_merged")

# Push to Hugging Face (uncomment and replace with your token)
# merged_model.push_to_hub("DestinBir/buzy-ai-gemma4", token="YOUR_HF_TOKEN")
# processor.push_to_hub("DestinBir/buzy-ai-gemma4", token="YOUR_HF_TOKEN")

print("Merge complete! Merged model saved to 'buzy_ai_gemma4_merged/'")
print("Uncomment the push_to_hub lines above to upload to Hugging Face.")


# Conclusion

Buzy AI démontre comment Gemma 4 peut être utilisé pour transformer des documents d'entreprise en connaissances exploitables et en recommandations décisionnelles explicables.

Le pipeline présenté combine :

- la compréhension multimodale ;
- l'extraction de connaissances ;
- le fine-tuning avec LoRA ;
- la recherche d'informations pertinentes ;
- le raisonnement métier structuré ;
- les capacités d'un agent IA.

Cette architecture constitue une base robuste pour le développement d'assistants décisionnels capables d'exploiter efficacement les connaissances d'une organisation.